## 03 - Data Anonymization

### Objective

This notebook prepares the extracted datasets for ethical analysis by removing personally identifiable information (PII).

Before generating anonymous student identifiers, the student records from the three academic terms are compared to verify that they represent the same cohort. This validation ensures that a single, consistent Student_ID can be assigned to each student and used across all datasets throughout the project.

The anonymized datasets produced in this notebook will be used for all subsequent data cleaning, analysis, SQL integration, and Power BI reporting.

In [ ]:
import pandas as pd

### Loading the Student Datasets

The Student datasets extracted from the three academic term workbooks are loaded for comparison.

In [ ]:
term1_students = pd.read_csv("../data/interim/term1/term1_student.csv")

term2_students = pd.read_csv("../data/interim/term2/term2_student.csv")

term3_students = pd.read_csv("../data/interim/term3/term3_student.csv")

### Inspecting the Student Datasets

The structure of each dataset is reviewed to confirm that the expected variables are available before comparing the student records.

In [ ]:
print("Term 1")
display(term1_students.head())

print("Term 2")
display(term2_students.head())

print("Term 3")
display(term3_students.head())

## Comparing the Number of Student Records

The number of student records in each academic term is compared to identify any differences in cohort size.

In [ ]:
comparison = pd.DataFrame({
    "Term": ["Term 1", "Term 2", "Term 3"],
    "Students": [
        len(term1_students),
        len(term2_students),
        len(term3_students)
    ]
})

comparison

In [ ]:
# Select only the columns required for comparison

term1_compare = term1_students[["STUDENT ID", "STUDENT NAME"]].copy()

term2_compare = term2_students[["STUDENT ID", "STUDENT NAME"]].copy()

term3_compare = term3_students[["STUDENT ID", "STUDENT NAME"]].copy()

In [ ]:
for df in [term1_compare, term2_compare, term3_compare]:

    df["STUDENT NAME"] = (
        df["STUDENT NAME"]
        .str.strip()
        .str.upper()
    )

In [ ]:
term1_vs_term2 = term1_compare.merge(
    term2_compare,
    on=["STUDENT ID", "STUDENT NAME"],
    how="outer",
    indicator=True
)

term1_vs_term2

In [ ]:
term1_vs_term2[
    term1_vs_term2["_merge"] != "both"
]

In [ ]:
term1_vs_term3 = term1_compare.merge(
    term3_compare,
    on=["STUDENT ID", "STUDENT NAME"],
    how="outer",
    indicator=True
)

term1_vs_term3[
    term1_vs_term3["_merge"] != "both"
]

In [ ]:
term2_vs_term3 = term2_compare.merge(
    term3_compare,
    on=["STUDENT ID", "STUDENT NAME"],
    how="outer",
    indicator=True
)

term2_vs_term3[
    term2_vs_term3["_merge"] != "both"
]

### Investigating Student Record Differences

The comparison identified differences between the student records across the three academic terms.

These differences may reflect changes in enrollment, such as withdrawals, or new admissions. Alternatively, they may indicate inconsistencies in data entry.

The unmatched records will be reviewed before generating the master student list.

## Creating a Master Student List

The student records from all three academic terms are combined into a single master list.

This ensures that every student who appeared during the academic year is represented before anonymization.

In [ ]:
master_students = pd.concat(
    [
        term1_compare,
        term2_compare,
        term3_compare
    ],
    ignore_index=True
)

master_students.head()

In [ ]:
master_students = (
    master_students
    .drop_duplicates(subset="STUDENT ID")
    .sort_values("STUDENT ID")
    .reset_index(drop=True)
)

master_students

In [ ]:
print("Total unique students:", len(master_students))

In [ ]:
master_students = (
    master_students
    .sort_values("STUDENT ID")
    .reset_index(drop=True)
)

master_students["Student_ID"] = [
    f"ST{str(i).zfill(4)}"
    for i in range(1, len(master_students) + 1)
]

In [ ]:
# Save the master student list to a CSV file
master_students.to_csv(
    "../data/metadata/student_lookup.csv",
    index=False
)

### Loading the Student Lookup Table

The student lookup table contains the mapping between the original school-assigned student identifiers and the anonymous Student_ID values thats was generated for this project.

This lookup table will be used to anonymize all extracted datasets while maintaining consistent relationships across the academic terms.

In [ ]:
student_lookup = pd.read_csv("../data/metadata/student_lookup.csv")

student_lookup.head()

### Creating a Reusable Anonymization Function

A reusable function is created to anonymize datasets containing student information.

The function performs the following tasks:

- Loads the dataset.
- Matches each student using the school-assigned STUDENT ID.
- Appends the anonymous Student_ID.
- Removes personally identifiable information.
- Saves the anonymized dataset for subsequent processing.

In [ ]:
def anonymize_dataset(df, lookup):

    if "STUDENT ID" in df.columns:

        anonymized = df.merge(
            lookup[["STUDENT ID", "Student_ID"]],
            on="STUDENT ID",
            how="left"
        )

    elif "STUDENT NAME" in df.columns:

        anonymized = df.merge(
            lookup[["STUDENT NAME", "Student_ID"]],
            on="STUDENT NAME",
            how="left"
        )

    elif "NAME" in df.columns:

        lookup_name = lookup.rename(
            columns={"STUDENT NAME": "NAME"}
        )

        anonymized = df.merge(
            lookup_name[["NAME", "Student_ID"]],
            on="NAME",
            how="left"
        )

    else:

        raise ValueError("No student identifier found.")

    anonymized.insert(
        0,
        "Student_ID",
        anonymized.pop("Student_ID")
    )

    return anonymized